# 球面と双曲面：$\varepsilon=\pm1$ でガウス曲率の符号が変わる

[ノート](../../../研究ノート/02_微分幾何/christoffel_riemann_intro.md#C-1-4)（付録C-1-4）の比較を、3Dで見ます。

どちらも、第二基本形式が $K_{ij}=-g_{ij}/a$ という同じ形ですが、

- **球面**（$\mathbb R^3$ の中、$\varepsilon=\bar g(n,n)=+1$）：$K_{\mathrm G}=+1/a^2$
- **双曲面**（ミンコフスキー空間 $(t,x,y)$ の中、$\varepsilon=-1$）：$K_{\mathrm G}=-1/a^2$

です。法線の長さの符号が、ガウス曲率の符号を決めています。

**使い方**：下部のスライダーで $a$ を変えます。3D の図は、ドラッグで回転、ホイールで拡大できます。

In [ ]:
import numpy as np
import sympy as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 検算：同じ手順で、2つの曲面の $\varepsilon$ と $K_{\mathrm G}$ を求める

どちらの曲面も、埋め込み $X(u,v)$ と、外側の計量 $G$（球面は単位行列、双曲面は $\mathrm{diag}(-1,1,1)$）を与えるだけで、次を SymPy が計算します。

1. 誘導計量 $g_{ij}=G(e_i,e_j)$、法線 $n=X/a$ と $\varepsilon=G(n,n)$
2. 外的曲率 $K_{ij}=G(\partial_i\partial_jX,\ n)$
3. ガウス方程式（外側が平坦）から $K_{\mathrm G}=\varepsilon\,\det K/\det g$
4. 計量だけからのクリストッフェル記号とリーマン曲率テンソルによる $K_{\mathrm G}$（独立な計算）

3 と 4 が一致すること、さらにハミルトン拘束 $\varepsilon R=(\operatorname{tr}K)^2-K_{ij}K^{ij}$（外側が平坦なので、$\bar G_{{nn}}=0$）が成り立つことを確かめます。

In [ ]:
a_ = sp.symbols("a", positive=True)
u, v = sp.symbols("u v", real=True)


def analyze(X, G, label):
    X = sp.Matrix(X)
    G = sp.Matrix(G)
    e = [X.diff(u), X.diff(v)]
    ip = lambda p, q: sp.simplify((p.T * G * q)[0])
    g = sp.Matrix(2, 2, lambda i, j: ip(e[i], e[j]))
    n = X / a_
    eps = ip(n, n)
    assert ip(n, e[0]) == 0 and ip(n, e[1]) == 0
    K = sp.Matrix(2, 2, lambda i, j: ip(X.diff([u, v][i]).diff([u, v][j]), n))
    KG_gauss = sp.simplify(eps * K.det() / g.det())
    # 計量だけから（Christoffel → Riemann）
    ginv, q = g.inv(), [u, v]
    Gam = [[[sp.simplify(sum(ginv[k, l] * (g[l, i].diff(q[j]) + g[l, j].diff(q[i]) - g[i, j].diff(q[l])) for l in range(2)) / 2)
             for j in range(2)] for i in range(2)] for k in range(2)]
    def Riem(l, k, i, j):
        t = Gam[l][j][k].diff(q[i]) - Gam[l][i][k].diff(q[j])
        t += sum(Gam[l][i][m] * Gam[m][j][k] - Gam[l][j][m] * Gam[m][i][k] for m in range(2))
        return sp.simplify(t)
    R1212 = sp.simplify(sum(g[0, m] * Riem(m, 1, 0, 1) for m in range(2)))
    KG_metric = sp.simplify(R1212 / g.det())
    Ric = sp.Matrix(2, 2, lambda k, j: sp.simplify(sum(Riem(i, k, i, j) for i in range(2))))
    R = sp.simplify(sum(ginv[i, j] * Ric[i, j] for i in range(2) for j in range(2)))
    S = ginv * K
    trK = sp.simplify(S.trace())
    KK = sp.simplify((S * S).trace())
    assert sp.simplify(KG_gauss - KG_metric) == 0
    assert sp.simplify(eps * R - (trK**2 - KK)) == 0
    print(f"{label}:  ε = {eps},  K_G = {KG_gauss}（ガウス方程式）= {KG_metric}（計量だけ）,  "
          f"R = {R},  tr K = {trK},  K_ij K^ij = {KK}")
    return eps, KG_gauss


th_, ph_, chi_ = sp.symbols("theta phi chi", real=True)
u, v = th_, ph_
eps_s, KG_s = analyze([a_ * sp.sin(u) * sp.cos(v), a_ * sp.sin(u) * sp.sin(v), a_ * sp.cos(u)], sp.eye(3), "球面（R^3）")
u, v = chi_, ph_
eps_h, KG_h = analyze([a_ * sp.cosh(u), a_ * sp.sinh(u) * sp.cos(v), a_ * sp.sinh(u) * sp.sin(v)], sp.diag(-1, 1, 1), "双曲面（ミンコフスキー空間）")
assert eps_s == 1 and eps_h == -1 and sp.simplify(KG_s - 1 / a_**2) == 0 and sp.simplify(KG_h + 1 / a_**2) == 0

## スライダーで動かす

左が球面（$\mathbb R^3$）、右が双曲面（ミンコフスキー空間。縦軸が時間 $t$、黄色は光円錐）です。紫の矢印は、点 $X$ での単位法線 $n=X/a$ です。

In [ ]:
a_vals = np.round(np.arange(0.6, 2.01, 0.1), 2)
ph = np.linspace(0, 2 * np.pi, 60)


def surfaces(a):
    uu, pp = np.meshgrid(np.linspace(0, np.pi, 40), ph, indexing="ij")
    S = (a * np.sin(uu) * np.cos(pp), a * np.sin(uu) * np.sin(pp), a * np.cos(uu))
    cc, pp = np.meshgrid(np.linspace(0, 1.0, 30), ph, indexing="ij")
    H = (a * np.sinh(cc) * np.cos(pp), a * np.sinh(cc) * np.sin(pp), a * np.cosh(cc))     # (x, y, t)
    return S, H


def normals(a):
    """双曲面・球面の上の数点での単位法線 n = X/a（長さ 0.6 の線分に描く）"""
    pts_s, pts_h = [], []
    for t_, p_ in [(0.9, 0.3), (1.6, 2.0), (2.3, 4.0)]:
        X = a * np.array([np.sin(t_) * np.cos(p_), np.sin(t_) * np.sin(p_), np.cos(t_)])
        pts_s.append((X, X + 0.6 * X / a))
    for c_, p_ in [(0.3, 0.3), (0.6, 2.0), (0.95, 4.0)]:
        X = a * np.array([np.sinh(c_) * np.cos(p_), np.sinh(c_) * np.sin(p_), np.cosh(c_)])
        pts_h.append((X, X + 0.6 * X / a))
    def gaps(rows):
        xs, ys, zs = [], [], []
        for p, q in rows:
            xs += [p[0], q[0], None]; ys += [p[1], q[1], None]; zs += [p[2], q[2], None]
        return xs, ys, zs
    return gaps(pts_s), gaps(pts_h)


def traces(a):
    (Sx, Sy, Sz), (Hx, Hy, Hz) = surfaces(a)
    ns, nh = normals(a)
    return [
        go.Surface(x=Sx, y=Sy, z=Sz, surfacecolor=np.ones_like(Sx), colorscale=[[0, "#8ec1e8"], [1, "#8ec1e8"]], showscale=False, opacity=0.85),
        go.Scatter3d(x=ns[0], y=ns[1], z=ns[2], mode="lines", line=dict(color="#7b2fbf", width=6), showlegend=False),
        go.Surface(x=Hx, y=Hy, z=Hz, surfacecolor=np.ones_like(Hx), colorscale=[[0, "#f0a35e"], [1, "#f0a35e"]], showscale=False, opacity=0.9),
        go.Scatter3d(x=nh[0], y=nh[1], z=nh[2], mode="lines", line=dict(color="#7b2fbf", width=6), showlegend=False),
    ]


# 光円錐（固定）
rr, pp = np.meshgrid(np.linspace(0, 3.4, 12), ph, indexing="ij")
cone_top = go.Surface(x=rr * np.cos(pp), y=rr * np.sin(pp), z=rr, opacity=0.15, showscale=False,
                      colorscale=[[0, "#f2d600"], [1, "#f2d600"]], hoverinfo="skip")

frames, steps = [], []
for a in a_vals:
    name = f"{a:.1f}"
    frames.append(go.Frame(data=traces(a), traces=[0, 1, 3, 4], name=name,
                           layout=go.Layout(title_text=f"a = {a:.1f}：球面 ε=+1, K_G=+{1 / a**2:.2f}　　双曲面 ε=−1, K_G=−{1 / a**2:.2f}")))
    steps.append(dict(method="animate", label=name,
                      args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))

k0 = int(np.argmin(np.abs(a_vals - 1.0)))
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=["球面（R³）", "双曲面（ミンコフスキー空間, 縦軸 t）"])
t0 = traces(a_vals[k0])
fig.add_trace(t0[0], row=1, col=1); fig.add_trace(t0[1], row=1, col=1)
fig.add_trace(cone_top, row=1, col=2)
fig.add_trace(t0[2], row=1, col=2); fig.add_trace(t0[3], row=1, col=2)
fig.frames = frames
rng_ = dict(range=[-2.8, 2.8], visible=False)
fig.update_layout(
    title_text=frames[k0].layout.title.text,
    sliders=[dict(active=k0, currentvalue=dict(prefix="a = "), pad=dict(t=30), steps=steps)],
    scene=dict(aspectmode="cube", xaxis=rng_, yaxis=rng_, zaxis=rng_, camera=dict(eye=dict(x=1.4, y=1.2, z=0.9))),
    scene2=dict(aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.7), xaxis=rng_, yaxis=rng_,
                zaxis=dict(range=[-0.3, 3.6], visible=False), camera=dict(eye=dict(x=1.5, y=1.3, z=0.8))),
    height=650, margin=dict(l=0, r=0, t=90, b=0),
)
fig.show()

## 観察のポイント

- 球面では、単位法線 $n=X/a$ は外向きで、$\bar g(n,n)=+1$ です。双曲面では、$n=X/a$ は時間方向（光円錐の内側）で、$\bar g(n,n)=-1$ です。
- 検算の出力のとおり、どちらも $K_{ij}=-g_{ij}/a$ の形ですが、ガウス方程式（C.5）の $KK$ の項に $\varepsilon$ が付くので、$K_{\mathrm G}$ の符号が変わります。
- $a$ を大きくすると、どちらも曲率 $|K_{\mathrm G}|=1/a^2$ が小さくなり、平面に近づきます。
- ハミルトン拘束（外側が平坦）の $\varepsilon R=(\operatorname{tr}K)^2-K_{ij}K^{ij}$ が、球面と双曲面の両方で成り立つことも、検算セルで確かめました。